In [27]:
!pip3 install torchinfo


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


# Lab 6: Neural Network (PyTorch)

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from ISLP import load_data
from ISLP.models import ModelSpec as MS
import torch
from torch import nn
from torch.utils.data import TensorDataset
from pytorch_lightning import Trainer
from ISLP.torch import SimpleDataModule, SimpleModule, ErrorTracker
from torchinfo import summary

## Question 3
Fit a neural network to the Default data. Use a single hidden layer with 10 units, and dropout
regularization. Have a look at Labs 10.9.1–10.9.2 for guidance.
### a. Split the data into training set (70%) and test set (30%)


In [19]:
default = load_data('Default')

In [20]:
default

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879
...,...,...,...,...
9995,No,No,711.555020,52992.378914
9996,No,No,757.962918,19660.721768
9997,No,No,845.411989,58636.156984
9998,No,No,1569.009053,36669.112365


In [21]:
y = np.where(default['default'] == 'Yes', 1, 0)

In [22]:
model = MS(default.columns.drop('default'), intercept=False)

In [23]:
X = model.fit_transform(default).to_numpy()

In [24]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.3, random_state=1)

### b. Compare the classification performance (test prediction accuracy) of your model with that of linear logistic regression based on the test set.

#### Standardize features

In [25]:
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

#### Fit linear logistic regression

In [26]:
logit = LogisticRegression()
logit.fit(X_train_s, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [27]:
logit_pred = logit.predict(X_test_s)
logit_acc = accuracy_score(Y_test, logit_pred)
print(f"Logistic Regression Prediction Accuracy: {logit_acc}")

Logistic Regression Prediction Accuracy: 0.9756666666666667


#### Fit neural networks
Use a single hidden layer with 10 units, and dropout regularization. 

In [28]:
# transform to be accessible to torch for both train and test
X_train_t = torch.tensor(X_train_s.astype(np.float32))
Y_train_t = torch.tensor(Y_train.astype(np.float32))
default_train = TensorDataset(X_train_t, Y_train_t)

X_test_t = torch.tensor(X_test_s.astype(np.float32))
Y_test_t = torch.tensor(Y_test.astype(np.float32))
default_test = TensorDataset(X_test_t, Y_test_t)

In [29]:
default_dm = SimpleDataModule(default_train, 
                              default_test,
                              batch_size = 32,
                              validation=default_test)

In [47]:
class DefaultModel(nn.Module):
    def __init__(self, input_size):
        super(DefaultModel, self).__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(10, 1))        
    def forward(self, x):
        return torch.flatten(self.sequential(x))

In [48]:
nn_model = DefaultModel(X_train_s.shape[1])

In [49]:
summary(nn_model ,
input_size =X_train_s.shape ,
col_names =['input_size','output_size','num_params'])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
DefaultModel                             [7000, 3]                 [7000]                    --
├─Sequential: 1-1                        [7000, 3]                 [7000, 1]                 --
│    └─Linear: 2-1                       [7000, 3]                 [7000, 10]                40
│    └─ReLU: 2-2                         [7000, 10]                [7000, 10]                --
│    └─Dropout: 2-3                      [7000, 10]                [7000, 10]                --
│    └─Linear: 2-4                       [7000, 10]                [7000, 1]                 11
Total params: 51
Trainable params: 51
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.36
Input size (MB): 0.08
Forward/backward pass size (MB): 0.62
Params size (MB): 0.00
Estimated Total Size (MB): 0.70

In [50]:
nn_module = SimpleModule.binary_classification(nn_model)

In [51]:
trainer = Trainer(deterministic=True, 
                  max_epochs=20, 
                  callbacks=[ErrorTracker()])

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [52]:
trainer.fit(nn_module, datamodule=default_dm)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name  | Type              | Params | Mode  | FLOPs
------------------------------------------------------------
0 | model | DefaultModel      | 51     | train | 0    
1 | loss  | BCEWithLogitsLoss | 0      | train | 0    
------------------------------------------------------------
51        Trainable params
0         Non-trainable params
51        Total params
0.000     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0: 100%|██████████████████████| 219/219 [00:00<00:00, 281.36it/s, v_num=4]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████████████████| 219/219 [00:00<00:00, 258.83it/s, v_num=4]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████████████████| 219/219 [00:00<00:00, 253.71it/s, v_num=4]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████████████████| 219/219 [00:00<00:00, 271.18it/s, v_num=4]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Validation: |                                             | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████████

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|█████████████████████| 219/219 [00:00<00:00, 243.26it/s, v_num=4]


In [53]:
nn_results = trainer.test(nn_module, datamodule=default_dm)

Testing DataLoader 0: 100%|███████████████████| 94/94 [00:00<00:00, 1020.40it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9696666598320007
        test_loss           0.08466586470603943
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


In [45]:
print(f"Neural Network Test Results: {nn_results}")

Neural Network Test Results: [{'test_loss': 0.6931472420692444, 'test_accuracy': 0.9696666598320007}]


In [46]:
print(f"Logistic Regression Prediction Accuracy: {logit_acc}")

Logistic Regression Prediction Accuracy: 0.9756666666666667


Based on the results, Logistics Regression slightly outperforms Neural Network at ~97.57% compared to ~97.43%, respectively.

This result can be explained by the bias-variance trade off. If the true relationship between features and the response is close to linear, a less flexible method would be a better fit as more complex model can have higher variance.